# ABI 7500 Fast Dx: offline hello world

```{device-card} thermo-fisher-abi7500-fast-dx
```

**Incomplete integration: no hardware control class is available.** This notebook decodes packets, reconstructs a captured camera frame, and reads saved vendor well values without communicating with an instrument. It does not run PCR or derive calibrated fluorescence from images. Direct PLR hardware operation remains unverified.

| Property | Value |
|---|---|
| Instrument connection | USB Type-B |
| Planned PLR transport | `pylabrobot.io.USB` |
| Current functionality | Offline packets, camera reconstruction, saved EDS filter arrays |
| Reference | 7500 Software v2.3 native routines and connection/acquisition/cancellation captures |
| PLR hardware validation | Pending |

See [the integration guide](index.md) and [protocol evidence](protocol.md) for limits and provenance.

## Connection and physical setup

The Hamamatsu USB connection carries camera messages and a tunnel for thermal-controller packets. An RS-232 EZ-Tap cannot capture this external USB link.

No cabling or device setup is needed for these offline cells. For future live sessions, use the original working control PC and [USB capture procedure](capture.md). Every connection or acquisition session requires its own explicit approval under `AGENTS.md`. Firmware changes and recalibration are outside scope.

## Import the offline codecs

These imports and all following cells operate on bytes only. There is no `setup()` call.

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx.protocol import (
  ThermalPacket,
  ThermalPacketDecoder,
  decode_serial_read,
  encode_serial_read,
  encode_serial_write,
)

## Decode a reference packet

This temperature-query packet was generated by the reference binary in an isolated CPU emulator. It is not a captured instrument response. The sequence value identifies a message, not an experiment run.

In [ ]:
reference = bytes.fromhex("5858ffffffff000f393730300006000000000001e6")
packet = ThermalPacket.decode(reference)
assert packet.command == 0x06
assert packet.sequence == 1
assert packet.payload == b""
assert packet.encode() == reference
packet

## Reassemble fragments

A thermal packet can span multiple tunnel replies. A strict decoder retains incomplete bytes and rejects malformed packets without guessing where the next one begins. `finish()` detects a truncated capture.

In [ ]:
decoder = ThermalPacketDecoder()
assert decoder.feed(reference[:9]) == []
reassembled = decoder.feed(reference[9:])
decoder.finish()
assert reassembled == [packet]
reassembled

## Inspect the USB serial wrapper

The recovered serial-write wrapper uses a four-character space-padded length and writes of at most 64 bytes. The serial-read request includes two bytes for the reply count. Inspect these bytes offline; the startup capture confirms the serial wrapper, but PLR hardware control remains unverified.

In [ ]:
write_chunks = encode_serial_write(reference)
read_request = encode_serial_read(len(reference))
assert write_chunks == (b"RS  21" + reference,)
assert read_request == b"RR  23"
{"write_hex": [chunk.hex() for chunk in write_chunks], "read_request": read_request}

## Separate valid bytes from the USB reply tail

Captured RR replies fill the requested length even when fewer serial bytes are available. The little-endian count identifies the valid bytes; the unused tail can be nonzero. This synthetic example wraps the earlier reference packet with four unused bytes. It illustrates parsing only, not a real instrument response. Unmodified captured examples are covered by the protocol tests.

In [ ]:
wire_reply = len(reference).to_bytes(2, "little") + reference + b"\xff" * 4
serial_bytes = decode_serial_read(wire_reply, requested_size=len(reference) + 4)
assert serial_bytes == reference
{"wire_bytes": len(wire_reply), "valid_serial_bytes": len(serial_bytes)}

## Reconstruct a captured camera frame

The repository contains one compressed CD payload as a regression fixture. Its pixels are little-endian unsigned 16-bit values. `CameraFrameAssembler` preserves every byte and reproduces the native library's crop, including its one-pixel offset. This example uses the original USB read boundaries; it does not identify physical wells or apply calibration.

In [ ]:
import base64
import json
import zlib
from importlib.resources import files

from pylabrobot.thermo_fisher.abi7500_fast_dx import CameraFrameAssembler, CameraGeometry

fixture = json.loads(
  files("pylabrobot.thermo_fisher.abi7500_fast_dx")
  .joinpath("test_data/camera_capture_vector.json")
  .read_text()
)
wire_data = zlib.decompress(base64.b64decode(fixture["payload"]))
geometry = CameraGeometry.decode(bytes.fromhex(fixture["ci_reply_hex"]))
assembler = CameraFrameAssembler(geometry)
offset = 0
for length in fixture["usb_read_lengths"]:
  assembler.feed(wire_data[offset : offset + length])
  offset += length
frame = assembler.finish()
assert frame.wire_data == wire_data
{"width": frame.width, "height": frame.height, "first_pixels": frame.pixel_rows()[0][:8]}

## Read saved vendor filter values

EDS members ending in `.tiff` can contain 96 signed 64-bit well values rather than camera TIFFs. This fixture comes from the completed run; its expected values were checked against the vendor's separate live result log. Indices and integer precision are preserved. These are vendor outputs whose optical processing must still be recovered, so they are not labeled uncorrected pixel sums or calibrated measurements.

For a complete local EDS file, import `read_eds_filter_data` from the same package and call `read_eds_filter_data("/path/to/experiment.eds")`. It returns the available arrays in acquisition order; it does not open the instrument.

In [ ]:
from pylabrobot.thermo_fisher.abi7500_fast_dx import SavedFilterData

saved_fixture = json.loads(
  files("pylabrobot.thermo_fisher.abi7500_fast_dx")
  .joinpath("test_data/saved_filter_vector.json")
  .read_text()
)
saved = SavedFilterData.decode(saved_fixture["member"], bytes.fromhex(saved_fixture["data_hex"]))
assert list(saved.well_values) == saved_fixture["expected_values"]
{
  "stage": saved.stage,
  "cycle": saved.cycle,
  "filter": saved.filter,
  "first_values": saved.well_values[:8],
}

## Remaining hardware work

The working installation, calibration files, and example experiments are backed up. A completed two-cycle run and a cancelled run supply 62 camera frames and matching vendor logs. Offline image bytes match the native reference copy, and all 960 completed-run well values match the live vendor result messages. Expired calibration warnings remain part of this evidence.

Exposure processing, well-mask geometry, calibration, run-state reconciliation, and live PLR transport/control remain unfinished. The hardware hello world must eventually cover async setup/stop, identity/status, explicit run identities, progress, cancellation, and original frames with raw/corrected measurements. See [the capture guide](capture.md).